# Governing with Artificial Intelligence
## Mapping the Knowledge Systems Shaping Urban Intelligence

**Desmond Lartey · Kris M.Y. Law**  
*Technology in Society* 86 (2026) 103321  
DOI: [10.1016/j.techsoc.2026.103321](https://doi.org/10.1016/j.techsoc.2026.103321)

---

### How to use this notebook

This notebook walks through the full analysis pipeline from the paper, one step at a time. Each section starts with a plain-language explanation of what is happening and why, followed by runnable code.

**You do not need to run every section** — figures in Sections 5 onward can be run independently once the data tables in Section 1 are defined.

**Input files** — place these in a folder called `data/` next to this notebook:

| File | Description |
|------|-------------|
| `Merged_Tagged_AIUrbanism.xlsx` | Master article database (one row per article) |
| `Hybrid_Conceptual_Lens_Weighted_Matrix.xlsx` | Lens weight scores per search indicator |
| `Hybrid_Conceptual_Lens_Weighted_Matrix_trend_contributing_Analysis.xlsx` | Yearly lens contribution totals |

All figures and result files are written to `output/`.

---
## Section 0 — Setup

Install and import all required libraries. The `sentence-transformers` library handles semantic text comparison; all other packages are standard scientific Python.

Run this cell first.

In [ ]:
# Uncomment the line below if you need to install packages
# !pip install pandas numpy matplotlib seaborn scikit-learn scipy statsmodels networkx sentence-transformers openpyxl

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx

from sentence_transformers import SentenceTransformer, util
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import MDS
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix
from statsmodels.nonparametric.smoothers_lowess import lowess
from matplotlib.patches import Ellipse

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# ── Folder configuration ──────────────────────────────────────────────────
# Change INPUT_DIR if your data files live somewhere else.
INPUT_DIR  = 'data'
OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def inp(filename):
    return os.path.join(INPUT_DIR, filename)

def out(filename):
    return os.path.join(OUTPUT_DIR, filename)

print('Setup complete.')
print('Input folder: ', os.path.abspath(INPUT_DIR))
print('Output folder:', os.path.abspath(OUTPUT_DIR))

---
## Section 1 — Conceptual Framework Definitions

Before loading any data, we define the conceptual scaffolding of the analysis.

The study organises AI urbanism literature around **12 search indicators** (e.g. Technology, Governance, Agency). These are grouped into **3 higher-order conceptual lenses**:

| Lens | What it covers |
|------|----------------|
| **Key Stakeholders and Entities** | Physical and institutional infrastructure — platforms, hardware, organisations |
| **Models of Interaction** | How AI and people interact — decision-making, feedback, co-agency |
| **External Influencing Factors** | Normative and political context — governance, ethics, regulation, culture |

The weights below quantify how strongly each indicator aligns with each lens, based on cosine similarity scoring (Figure 3b in the paper).

In [ ]:
# ── Indicator to lens mapping (Table 2 in the paper) ─────────────────────
INDICATOR_TO_LENS = {
    'Technology':    'Key Stakeholders and entities',
    'Action':        'Models of Interaction',
    'Space':         'Models of Interaction',
    'Agency':        'Models of Interaction',
    'Culture':       'Models of Interaction',
    'Personality':   'External influencing factors',
    'Time':          'External influencing factors',
    'Materiality':   'Key Stakeholders and entities',
    'Data':          'External influencing factors',
    'Governance':    'External influencing factors',
    'Sustainability':'Key Stakeholders and entities',
    'Security':      'External influencing factors',
}

# ── Lens descriptions used as semantic reference vectors ──────────────────
LENS_DESCRIPTIONS = {
    'Key Stakeholders and entities': (
        'Infrastructure, sensors, IoT, AIoT, platforms, and the urban systems '
        'or organisations enabling AI urbanism'
    ),
    'Models of Interaction': (
        'Interaction, decision-making, feedback loops, predictive analytics, '
        'and how AI behaves in the urban environment'
    ),
    'External influencing factors': (
        'Governance, policies, ethics, regulation, sustainability, cultural '
        'context, and normative factors shaping AI urbanism'
    ),
}

# ── Consistent colour palette used across all figures ─────────────────────
LENS_COLORS = {
    'Key Stakeholders and entities': '#e74c3c',
    'Models of Interaction':         '#3498db',
    'External influencing factors':  '#2ecc71',
}

print('Lens definitions loaded.')

In [ ]:
# ── Weighted lens scores per indicator (Figure 3b values) ─────────────────
LENS_WEIGHTS = {
    'SearchIndicator': [
        'Action', 'Agency', 'Culture', 'Data', 'Governance',
        'Materiality', 'Personality', 'Security', 'Space',
        'Sustainability', 'Technology', 'Time'
    ],
    'Key Stakeholders and entities': [
        3.184, 11.406, 11.814, 34.658, 17.654,
        5.784, 3.661, 2.402, 22.788, 54.695, 59.124, 4.803
    ],
    'Models of Interaction': [
        5.166, 2.225, 11.035, 16.032, 2.046,
        4.252, 1.122, 0.585, 8.165, 17.33, 19.398, 2.679
    ],
    'External influencing factors': [
        1.179, 35.071, 9.036, 29.109, 52.921,
        4.241, 6.406, 1.171, 10.369, 24.763, 17.346, 2.824
    ],
}

df_weights = pd.DataFrame(LENS_WEIGHTS).set_index('SearchIndicator')
print('Lens weight matrix — shape:', df_weights.shape)
df_weights

In [ ]:
# ── Yearly lens contribution data (Figure 4b values) ─────────────────────
TEMPORAL_DATA = {
    'Year': [1990, 2006, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'Key Stakeholders and entities':  [0, 0, 18, 49, 80, 84, 147, 195, 282, 338, 393, 805, 205],
    'Models of Interaction':          [1, 1, 34, 50, 73, 102, 137, 180, 269, 374, 434, 903, 265],
    'External influencing factors':   [0, 0, 10, 22, 46, 47, 56, 82, 169, 194, 266, 586, 167],
}

# ── Subsystem connection data (Figure 6 values) ───────────────────────────
SUBSYSTEM_DATA = {
    'Subsystem': [
        'Decision-Making Dynamics', 'Citizen Participation Models', 'Human-AI Interfaces',
        'Urban Data Infrastructure', 'AI Hardware & Platforms', 'Institutional Stakeholders',
        'Platform Governance', 'Ethical Standards', 'Accountability Mechanisms',
        'Legal/Policy Instruments', 'Feedback Loops', 'Public Legitimacy'
    ],
    'Key Stakeholders and entities': [30, 5, 10, 40, 35, 25, 25, 10, 5, 10, 5, 15],
    'Models of Interaction':         [20, 40, 35, 10, 10, 5, 20, 20, 25, 15, 35, 20],
    'External influencing factors':  [10, 25, 15, 10, 10, 5, 15, 45, 50, 40, 35, 30],
}

print('All data tables defined.')

---
## Section 2 — Hybrid Article Classification

This section assigns each article to one of the three lenses using a two-stage hybrid approach (Section 3.1.2 of the paper).

### Stage 1 — Keyword matching
Each article already has a `SearchIndicator` column from the original search strategy. A lookup table maps each indicator directly to its primary lens. This is fast and deterministic.

### Stage 2 — Sentence-BERT semantic similarity
For articles with ambiguous or missing keyword matches, the article text (title + abstract + keywords) is encoded into a dense vector using the `all-MiniLM-L6-v2` language model. We compute cosine similarity against short descriptions of each lens. Matches above 0.45 are accepted; the highest-scoring lens wins.

### Ensemble rule
The keyword-based assignment takes priority. The semantic signal fills any remaining gaps.

> **Note:** Encoding ~5,600 articles takes 3-5 minutes on a standard laptop.

In [ ]:
# Load the master article database
# Required columns: Title, Abstract, Keyword, Year, SearchIndicator
master_file = inp('Merged_Tagged_AIUrbanism.xlsx')

df = pd.read_excel(master_file, dtype=str).fillna('')
df.columns = df.columns.str.strip()
df['SearchIndicator'] = df['SearchIndicator'].str.strip()

print(f'Articles loaded: {len(df):,}')
print('Columns detected:', list(df.columns))
df.head(3)

In [ ]:
# Stage 1: keyword-based lens assignment
df['Keyword_Lens'] = df['SearchIndicator'].map(INDICATOR_TO_LENS).fillna('Unclassified')

print('Keyword-based lens distribution:')
print(df['Keyword_Lens'].value_counts())

In [ ]:
# Stage 2: semantic similarity via Sentence-BERT
# The model is downloaded automatically on first use.
model = SentenceTransformer('all-MiniLM-L6-v2')

lens_names      = list(LENS_DESCRIPTIONS.keys())
lens_embeddings = model.encode(list(LENS_DESCRIPTIONS.values()), convert_to_tensor=True)

SIMILARITY_THRESHOLD = 0.45

def compute_semantic_lens(text):
    """Return the best-matching lens for a piece of text.
    Returns Unclassified if nothing clears the similarity threshold."""
    if not text.strip():
        return 'Unclassified'
    embedding = model.encode(text, convert_to_tensor=True)
    scores    = util.cos_sim(embedding, lens_embeddings)[0].tolist()
    best      = max(scores)
    if best < SIMILARITY_THRESHOLD:
        return 'Unclassified'
    return lens_names[scores.index(best)]

# Combine title, abstract, and keywords into a single text field per article
combined_text = (
    df.get('Title',    pd.Series([''] * len(df))) + ' ' +
    df.get('Abstract', pd.Series([''] * len(df))) + ' ' +
    df.get('Keyword',  pd.Series([''] * len(df)))
)

print('Running semantic classification — this may take a few minutes...')
df['Semantic_Lens'] = combined_text.apply(compute_semantic_lens)
print('Done.')

In [ ]:
# Ensemble: keyword match wins; semantic fills gaps
df['ConceptualLens'] = df.apply(
    lambda row: row['Keyword_Lens']
    if row['Keyword_Lens'] != 'Unclassified'
    else row['Semantic_Lens'],
    axis=1,
)

# Save the tagged database
tagged_file = out('Tagged_AIUrbanism.xlsx')
df.to_excel(tagged_file, index=False)

print('Final lens distribution:')
print(df['ConceptualLens'].value_counts())
print(f'\nSaved to: {tagged_file}')

---
## Section 3 — Matrix Construction

We build two summary tables that feed the ordination and temporal analyses.

**Weighted matrix** — rows are search indicators, columns are lenses. Each cell is the number of articles with that indicator assigned to that lens. This captures the cross-lens footprint of each indicator.

**Trend matrix** — rows are years, columns are lenses. Each cell is the total number of articles published in that year assigned to that lens.

In [ ]:
# Weighted matrix: indicator x lens article counts
pivot = df.pivot_table(
    index='SearchIndicator',
    columns='ConceptualLens',
    aggfunc='size',
    fill_value=0,
).reset_index()

pivot.to_excel(out('Weighted_Matrix.xlsx'), index=False)
print('Weighted matrix saved.')
pivot

In [ ]:
# Trend matrix: year x lens article counts
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df_year = df.dropna(subset=['Year', 'ConceptualLens']).copy()
df_year['Year'] = df_year['Year'].astype(int)

trend_matrix = df_year.pivot_table(
    index='Year',
    columns='ConceptualLens',
    aggfunc='size',
    fill_value=0,
).reset_index()

trend_matrix.to_excel(out('Trend_Matrix.xlsx'), index=False)
print('Trend matrix saved.')
trend_matrix.tail(8)

---
## Section 4 — Validation

To verify that the automated classification is reliable, Section 3.1.3 of the paper uses a stratified 10% sample that a human coder labels independently. Cohen's kappa then measures agreement between automated and human labels.

**Step 1:** Run the first cell to generate `Validation_Sample.xlsx`. It will have a blank `Manual_Lens` column.

**Step 2:** Open the file, fill in your own lens label for each row (choose from the three lens names used throughout), then return here and run the scoring cell.

The paper reports **95.74% exact agreement** and **kappa = 0.933**.

In [ ]:
# Create the stratified validation sample spreadsheet
validation_file = out('Validation_Sample.xlsx')

df_val = df[['Title', 'Keyword', 'Abstract', 'SearchIndicator', 'ConceptualLens']].copy()
df_val = df_val.rename(columns={'ConceptualLens': 'Auto_Lens'})

# 10% from each indicator group
parts = []
for _, group in df_val.groupby('SearchIndicator'):
    n = max(1, round(len(group) * 0.10))
    parts.append(group.sample(n=n, random_state=42))

validation = pd.concat(parts).drop_duplicates().reset_index(drop=True)
validation['Manual_Lens'] = ''
validation['Notes']       = ''

validation.to_excel(validation_file, index=False)
print(f'Validation sample: {len(validation)} rows')
print(f'Saved to: {validation_file}')
print('\nFill in Manual_Lens column, then run the scoring cell below.')
validation[['Title', 'SearchIndicator', 'Auto_Lens']].head(5)

In [ ]:
# Score the validation once Manual_Lens is filled in
df_coded = pd.read_excel(validation_file).fillna('')
coded    = df_coded[df_coded['Manual_Lens'].str.strip() != ''].copy()

if coded.empty:
    print('No manually coded rows found yet. Fill in the Manual_Lens column first.')
else:
    agreement = (
        coded['Auto_Lens'].str.strip() == coded['Manual_Lens'].str.strip()
    ).mean() * 100
    kappa = cohen_kappa_score(
        coded['Auto_Lens'].str.strip(),
        coded['Manual_Lens'].str.strip(),
    )
    print(f'Rows coded:      {len(coded)}')
    print(f'Exact agreement: {agreement:.2f}%')
    print(f"Cohen's kappa:   {kappa:.3f}")
    print('\nClassification report:')
    print(classification_report(coded['Manual_Lens'], coded['Auto_Lens']))

---
## Section 5 — Figure 2: Indicator Trends and Growth Dynamics

This figure answers **Research Question 1**: what are the dominant and emerging conceptual categories in AI urbanism, and how have they evolved over time?

Four panels show different angles on publication trends:

- **a** — raw annual counts per indicator as line charts
- **b** — distribution of counts grouped by five-year period as a boxplot
- **c** — period-level trajectories per indicator
- **d** — a bubble chart classifying each indicator as Dominant, Emerging, Declining, or Marginal, based on total publications (x-axis) and growth rate (y-axis)

Key finding: Technology and Action are dominant. Personality and Agency are rapidly emerging. Data, Security, and Materiality are declining in relative prominence.

In [ ]:
df_fig2 = pd.read_excel(inp('Merged_Tagged_AIUrbanism.xlsx'))
df_fig2 = df_fig2[['SearchIndicator', 'Year']].dropna()
df_fig2['SearchIndicator'] = df_fig2['SearchIndicator'].str.strip()
df_fig2['Year'] = df_fig2['Year'].astype(str).str.extract(r'(\d{4})').astype(int)

annual = df_fig2.groupby(['Year', 'SearchIndicator']).size().reset_index(name='Count')

bins   = [2009, 2012, 2015, 2018, 2020, 2022, 2025]
labels = ['2010-12', '2013-15', '2016-18', '2019-20', '2021-22', '2023-25']
df_fig2['Period'] = pd.cut(df_fig2['Year'], bins=bins, labels=labels, include_lowest=True)
period_counts = df_fig2.groupby(['Period', 'SearchIndicator']).size().reset_index(name='Count')

print('Data prepared. Annual records:', len(annual))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel a: annual trend lines
ax = axes[0, 0]
for indicator, group in annual.groupby('SearchIndicator'):
    ax.plot(group['Year'], group['Count'], marker='o', markersize=3, label=indicator)
ax.set_title('a) Annual publication counts per search indicator', fontsize=11)
ax.set_xlabel('Year')
ax.set_ylabel('Number of papers')
ax.legend(fontsize=7, ncol=2)

# Panel b: boxplot by time period
ax = axes[0, 1]
sns.boxplot(data=period_counts, x='SearchIndicator', y='Count',
            hue='Period', ax=ax, palette='Set2')
ax.set_title('b) Indicator frequency by five-year period', fontsize=11)
ax.set_xlabel('Search indicator')
ax.set_ylabel('Frequency')
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=7, title='Period')

# Panel c: period trajectory lines
ax = axes[1, 0]
for indicator, group in period_counts.groupby('SearchIndicator'):
    ax.plot(group['Period'].astype(str), group['Count'], marker='o', markersize=4, label=indicator)
ax.set_title('c) Temporal frequency trajectories across periods', fontsize=11)
ax.set_xlabel('Period')
ax.set_ylabel('Frequency')
ax.tick_params(axis='x', rotation=30)
ax.legend(fontsize=7, ncol=2)

# Panel d: dominant / emerging bubble chart
ax = axes[1, 1]
summary = annual.groupby('SearchIndicator').agg(TotalPubs=('Count', 'sum'))
first_cnt = annual.groupby('SearchIndicator').first()['Count']
last_cnt  = annual.groupby('SearchIndicator').last()['Count']
first_yr  = df_fig2.groupby('SearchIndicator')['Year'].min()
last_yr   = df_fig2.groupby('SearchIndicator')['Year'].max()
summary['Growth'] = ((last_cnt - first_cnt) / (last_yr - first_yr + 1)).fillna(0)

ax.scatter(summary['TotalPubs'], summary['Growth'],
           s=summary['TotalPubs'] / 2, alpha=0.7,
           c=range(len(summary)), cmap='tab20')
for ind, row in summary.iterrows():
    ax.annotate(ind, (row['TotalPubs'], row['Growth']), fontsize=8)

med_t = summary['TotalPubs'].median()
med_g = summary['Growth'].median()
ax.axvline(med_t, color='grey', linestyle='--', alpha=0.6)
ax.axhline(med_g, color='grey', linestyle='--', alpha=0.6)
for label, xpos, ypos in [
    ('Dominant',  med_t + 10, med_g + 1),
    ('Emerging',  10,         med_g + 1),
    ('Marginal',  10,         med_g - 5),
    ('Declining', med_t + 10, med_g - 5),
]:
    ax.text(xpos, ypos, label, fontsize=9, color='grey')
ax.set_title('d) Dominant vs. Emerging vs. Declining indicators', fontsize=11)
ax.set_xlabel('Total publications')
ax.set_ylabel('Growth rate (papers per year)')

plt.tight_layout()
plt.savefig(out('figure2_indicator_trends.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

---
## Section 6 — Figure 3: Semantic Mapping of Conceptual Lenses

This figure answers **Research Question 2**: how do the 12 indicators cluster into broader cognitive lenses, and what governance logics do those lenses encode?

Three panels:

- **b** — heatmap: how strongly each indicator aligns with each lens (darker = stronger)
- **c** — network: each indicator connects to each lens; edge thickness reflects alignment strength
- **d** — stacked bar: what proportion of each indicator's total semantic weight falls within each lens

Key finding: Technology and Sustainability are almost entirely within the Key Stakeholders lens. Governance and Data are strongly External Influencing Factors. Culture and Agency are more evenly distributed across lenses — they are inherently cross-cutting.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Panel b: heatmap
ax = axes[0]
sns.heatmap(df_weights, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('b) Weighted mapping: indicators to lenses', fontsize=11)
ax.set_xlabel('Conceptual lens')
ax.set_ylabel('Search indicator')
ax.tick_params(axis='x', rotation=30)

# Panel c: semantic network
ax = axes[1]
G = nx.Graph()
lens_names = list(LENS_COLORS.keys())
for lens in lens_names:
    G.add_node(lens, node_type='lens')
for ind in df_weights.index:
    G.add_node(ind, node_type='indicator')
    for lens in lens_names:
        w = df_weights.loc[ind, lens]
        if w > 0:
            G.add_edge(ind, lens, weight=w)

pos = nx.spring_layout(G, seed=42, k=2)
node_colors = [
    LENS_COLORS[n] if G.nodes[n]['node_type'] == 'lens' else '#95a5a6'
    for n in G.nodes()
]
edge_widths = [d['weight'] / 20 for _, _, d in G.edges(data=True)]
nx.draw_networkx(
    G, pos, ax=ax,
    node_color=node_colors,
    node_size=[700 if G.nodes[n]['node_type'] == 'lens' else 300 for n in G.nodes()],
    edge_color='grey', width=edge_widths, font_size=7, with_labels=True,
)
ax.set_title('c) Semantic network: indicators and lenses', fontsize=11)
ax.axis('off')

# Panel d: proportional stacked bar
ax = axes[2]
df_prop = df_weights.div(df_weights.sum(axis=1), axis=0)
df_prop.plot(kind='bar', stacked=True, ax=ax, color=list(LENS_COLORS.values()))
ax.set_title('d) Proportional lens contribution per indicator', fontsize=11)
ax.set_xlabel('Search indicator')
ax.set_ylabel('Proportion')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Lens', fontsize=7, bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.savefig(out('figure3_heatmap_network.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

---
## Section 7 — Figure 4: Temporal Evolution of the Three Lenses

This panel tracks how the relative prominence of each knowledge lens has shifted from 1990 to 2025.

We use **LOWESS smoothing** (locally weighted scatterplot smoothing) rather than a moving average. LOWESS adapts to local data density, which makes it better suited here because the corpus is sparse in the early years (pre-2010) and grows rapidly after 2018.

Key finding: the **Key Stakeholders** lens plateaus after 2019, suggesting epistemic saturation of the infrastructure-focused discourse. Meanwhile **External Influencing Factors** accelerates sharply post-2020, reflecting the rise of AI ethics, data governance, and normative concerns.

In [ ]:
df_temporal = pd.DataFrame(TEMPORAL_DATA).sort_values('Year')
lens_columns = list(LENS_COLORS.keys())

fig, ax = plt.subplots(figsize=(12, 6))

for lens in lens_columns:
    color = LENS_COLORS[lens]
    ax.scatter(df_temporal['Year'], df_temporal[lens], color=color, alpha=0.4, s=30)
    smoothed = lowess(df_temporal[lens], df_temporal['Year'], frac=0.4)
    ax.plot(smoothed[:, 0], smoothed[:, 1], color=color, linewidth=2.5, label=lens)

ax.set_title('Temporal evolution of conceptual lenses in AI Urbanism (LOWESS-smoothed)', fontsize=12)
ax.set_xlabel('Year')
ax.set_ylabel('Weighted contribution (article count)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(out('figure4_temporal_lowess.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

---
## Section 8 — Figure 5: Ordination Analysis

This section provides the statistical evidence that the three lenses are genuinely distinct regions of conceptual space (Section 3.2.2 of the paper).

| Method | Purpose |
|--------|---------|
| **PCA** | Projects temporal data to 2D; shows the direction of change over decades |
| **NMDS** | Positions indicators in 2D by preserving rank-order distances from the lens weight profile |
| **GNMDS** | Metric MDS — uses actual distances rather than ranks; observer-independent cross-check |

The **NMDS stress value** (printed when running) indicates how well the 2D layout captures the true structure. Values below 0.2 are considered acceptable; below 0.1 is excellent.

The **cluster ellipses** in Panel c are standard deviation ellipses drawn around each lens group. Clear separation between ellipses supports the three-lens structure.

In [ ]:
# Load the pre-computed weighted matrix (preferred) or use the one built in Section 3
weighted_file = inp('Hybrid_Conceptual_Lens_Weighted_Matrix.xlsx')
df_weighted   = pd.read_excel(weighted_file).set_index('SearchIndicator')

trend_file = inp('Hybrid_Conceptual_Lens_Weighted_Matrix_trend_contributing_Analysis.xlsx')
df_trend   = pd.read_excel(trend_file)
df_trend.rename(columns={df_trend.columns[0]: 'Year'}, inplace=True)
df_trend.set_index('Year', inplace=True)
df_trend.dropna(inplace=True)

print('Weighted matrix:', df_weighted.shape)
print('Trend matrix:   ', df_trend.shape)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Panel a: PCA biplot of temporal trajectories
ax = axes[0, 0]
scaler       = StandardScaler()
trend_scaled = scaler.fit_transform(df_trend)
pca          = PCA(n_components=2)
comps        = pca.fit_transform(trend_scaled)
expl         = pca.explained_variance_ratio_

era_color = {
    y: '#e74c3c' if y <= 2014 else '#3498db' if y <= 2019 else '#2ecc71'
    for y in df_trend.index
}
for i, year in enumerate(df_trend.index):
    ax.scatter(comps[i, 0], comps[i, 1], color=era_color[year], s=60, zorder=4)
    ax.annotate(str(year), (comps[i, 0], comps[i, 1]), fontsize=8)
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')
ax.set_title(f'a) PCA biplot of temporal lens trajectories\nPC1={expl[0]:.1%}, PC2={expl[1]:.1%}', fontsize=10)
ax.set_xlabel(f'PC1 ({expl[0]:.1%})')
ax.set_ylabel(f'PC2 ({expl[1]:.1%})')

# Compute NMDS (shared between panels b and c)
scaler2         = StandardScaler()
weighted_scaled = scaler2.fit_transform(df_weighted)
diss            = pairwise_distances(weighted_scaled, metric='euclidean')
mds             = MDS(n_components=2, dissimilarity='precomputed',
                      random_state=42, metric=False, max_iter=3000)
nmds_coords     = mds.fit_transform(diss)
print(f'NMDS stress: {mds.stress_:.4f}  (below 0.2 is acceptable)')

dominant_lens  = df_weighted.idxmax(axis=1)
legend_handles = [mpatches.Patch(color=c, label=l) for l, c in LENS_COLORS.items()]

# Panel b: NMDS coloured by dominant lens
ax = axes[0, 1]
for i, ind in enumerate(df_weighted.index):
    color = LENS_COLORS.get(dominant_lens[ind], '#95a5a6')
    ax.scatter(nmds_coords[i, 0], nmds_coords[i, 1], color=color, s=80, zorder=3)
    ax.annotate(ind, (nmds_coords[i, 0], nmds_coords[i, 1]), fontsize=8)
ax.legend(handles=legend_handles, fontsize=7)
ax.set_title('b) NMDS: indicators positioned by lens weight', fontsize=10)
ax.set_xlabel('NMDS Dimension 1')
ax.set_ylabel('NMDS Dimension 2')

# Panel c: NMDS with cluster ellipses
ax = axes[1, 0]
for i, ind in enumerate(df_weighted.index):
    color = LENS_COLORS.get(dominant_lens[ind], '#95a5a6')
    ax.scatter(nmds_coords[i, 0], nmds_coords[i, 1], color=color, s=80, zorder=3)
    ax.annotate(ind, (nmds_coords[i, 0], nmds_coords[i, 1]), fontsize=8)

for lens, color in LENS_COLORS.items():
    idx = [i for i, ind in enumerate(df_weighted.index) if dominant_lens[ind] == lens]
    if len(idx) < 2:
        continue
    pts = nmds_coords[idx]
    cx, cy = pts.mean(axis=0)
    cov = np.cov(pts.T)
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    ell = Ellipse(
        (cx, cy),
        width=2 * np.sqrt(max(vals[0], 1e-6)),
        height=2 * np.sqrt(max(vals[1], 1e-6)),
        angle=angle,
        edgecolor=color, facecolor='none',
        linewidth=1.5, linestyle='--', alpha=0.8,
    )
    ax.add_patch(ell)
ax.legend(handles=legend_handles, fontsize=7)
ax.set_title('c) NMDS with lens cluster ellipses', fontsize=10)
ax.set_xlabel('NMDS Dimension 1')
ax.set_ylabel('NMDS Dimension 2')

# Panel d: GNMDS (metric MDS)
ax = axes[1, 1]
mds_metric   = MDS(n_components=2, dissimilarity='precomputed',
                   random_state=42, metric=True, max_iter=3000)
gnmds_coords = mds_metric.fit_transform(diss)
for i, ind in enumerate(df_weighted.index):
    color = LENS_COLORS.get(dominant_lens[ind], '#95a5a6')
    ax.scatter(gnmds_coords[i, 0], gnmds_coords[i, 1], color=color, s=80, zorder=3)
    ax.annotate(ind, (gnmds_coords[i, 0], gnmds_coords[i, 1]), fontsize=8)
ax.legend(handles=legend_handles, fontsize=7)
ax.set_title('d) GNMDS: observer-independent ordination', fontsize=10)
ax.set_xlabel('GNMDS Dimension 1')
ax.set_ylabel('GNMDS Dimension 2')

plt.tight_layout()
plt.savefig(out('figure5_ordination.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure 5 saved.')

---
## Section 9 — Figure 6: Systems-Based Governance Framework

This is the integrative centrepiece of the paper, answering **Research Question 3**: how can the three knowledge lenses be synthesised into a governance framework?

The figure is a weighted network where:

- **Three coloured hub nodes** at the centre represent the conceptual lenses
- **Twelve grey peripheral nodes** represent urban governance subsystems
- **Edge thickness** reflects how strongly each lens influences each subsystem

The layout shows that each subsystem is shaped by all three lenses simultaneously. For example, Ethical Standards is mostly External Influencing Factors, but also has a smaller Models of Interaction component — because ethics is not purely regulatory but also enacted through human-AI interactions.

This network is the basis for the paper's practical recommendations: cities that treat ethics as purely external (green edges only) miss the interactional dimension, and vice versa.

In [ ]:
df_sub     = pd.DataFrame(SUBSYSTEM_DATA)
lens_names = list(LENS_COLORS.keys())

# Build the graph
G = nx.Graph()
for lens in lens_names:
    G.add_node(lens, node_type='lens')
for sub in df_sub['Subsystem']:
    G.add_node(sub, node_type='subsystem')
for _, row in df_sub.iterrows():
    for lens in lens_names:
        w = row[lens]
        if w > 0:
            G.add_edge(row['Subsystem'], lens, weight=w)

# Lenses at centre, subsystems evenly around a circle
pos = {}
n_sub = len(df_sub)
for i, sub in enumerate(df_sub['Subsystem']):
    angle = 2 * 3.14159 * i / n_sub
    pos[sub] = (__import__('math').cos(angle) * 3, __import__('math').sin(angle) * 3)
pos['Key Stakeholders and entities'] = ( 0.8,  0.5)
pos['Models of Interaction']         = (-0.8,  0.5)
pos['External influencing factors']  = ( 0.0, -0.8)

node_colors = [
    LENS_COLORS[n] if G.nodes[n]['node_type'] == 'lens' else '#bdc3c7'
    for n in G.nodes()
]
edge_colors, edge_widths = [], []
for u, v, d in G.edges(data=True):
    lens_node = u if u in LENS_COLORS else v
    edge_colors.append(LENS_COLORS.get(lens_node, '#95a5a6'))
    edge_widths.append(d['weight'] / 15)

fig, ax = plt.subplots(figsize=(14, 12))
nx.draw_networkx(
    G, pos, ax=ax,
    node_color=node_colors,
    node_size=[1500 if G.nodes[n]['node_type'] == 'lens' else 600 for n in G.nodes()],
    edge_color=edge_colors, width=edge_widths,
    font_size=7, with_labels=True, alpha=0.9,
)
legend_handles = [mpatches.Patch(color=c, label=l) for l, c in LENS_COLORS.items()]
legend_handles.append(mpatches.Patch(color='#bdc3c7', label='Urban subsystem'))
ax.legend(handles=legend_handles, fontsize=8, loc='lower right')
ax.set_title(
    'Systems-based governance framework of AI Urbanism\n'
    'Conceptual lenses (coloured) and urban governance subsystems (grey)',
    fontsize=13,
)
ax.axis('off')
plt.tight_layout()
plt.savefig(out('figure6_governance_framework.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure 6 saved.')

---
## Section 10 — Output Checklist

Run the cell below to verify which output files have been produced.

In [ ]:
expected = [
    'Tagged_AIUrbanism.xlsx',
    'Weighted_Matrix.xlsx',
    'Trend_Matrix.xlsx',
    'Validation_Sample.xlsx',
    'figure2_indicator_trends.png',
    'figure3_heatmap_network.png',
    'figure4_temporal_lowess.png',
    'figure5_ordination.png',
    'figure6_governance_framework.png',
]

print(f'Output folder: {os.path.abspath(OUTPUT_DIR)}\n')
for fname in expected:
    status = 'found           ' if os.path.exists(out(fname)) else 'not yet produced'
    print(f'  {status}  {fname}')

---

### Citation

If you use this code in your own work, please cite:

```
Lartey, D. & Law, K.M.Y. (2026). Governing with artificial intelligence:
Mapping the knowledge systems shaping urban intelligence.
Technology in Society, 86, 103321.
https://doi.org/10.1016/j.techsoc.2026.103321
```